# LECTURE 18 — HANDS-ON EXERCISE
### MODULE 7: UNSUPERVISED LEARNING & DIMENSIONALITY REDUCTION

**HANDS-ON EXERCISE** — Rank African Economies Using PCA-Derived Resilience Indicators

*WAIFEM ML Facilitation Programme 2026 — Compatible with Google Colab & VS Code*

## ── SECTION 1: IMPORTS

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120})

try:
    import google.colab          # noqa: F401
    print("Environment : Google Colab")
except ImportError:
    print("Environment : Local (VS Code / terminal)")

print("Libraries loaded successfully.\n")

## ── SECTION 2: FISCAL VULNERABILITY DATA (provided)

In [ ]:
COUNTRIES = [
    'Nigeria',        'South Africa',   'Egypt',          'Kenya',
    'Ethiopia',       'Ghana',          'Tanzania',       "Cote d'Ivoire",
    'Cameroon',       'Angola',         'Uganda',         'Mozambique',
    'Zambia',         'Zimbabwe',       'Senegal',        'Mali',
    'Burkina Faso',   'Guinea',         'Niger',          'Rwanda',
    'Benin',          'Burundi',        'Sierra Leone',   'Togo',
    'Libya',          'Tunisia',        'Algeria',        'Morocco',
    'Sudan',          'Madagascar',     'Botswana',       'Namibia',
    'Mauritius',      'Malawi',         'DR Congo',       'Congo Republic',
    'Gabon',          'Eq. Guinea',     'Eswatini',       'Lesotho',
    'Djibouti',       'Eritrea',        'Somalia',        'Chad',
    'Cabo Verde',
]
N = len(COUNTRIES)


def generate_fiscal_data(countries: list, seed: int = 55) -> pd.DataFrame:
    """
    Synthetic fiscal vulnerability indicators for 45 African economies.

    Indicators:
      debt_to_gdp          — government debt (% GDP)
      fiscal_deficit       — overall fiscal balance (% GDP), negative = deficit
      interest_to_revenue  — interest payments (% government revenue)
      external_debt_share  — external debt as % of total debt
      revenue_to_gdp       — government revenue (% GDP)
      grants_dependency    — grants as % of government revenue
      debt_service_ratio   — debt service (% exports)
      contingent_liabilities— SOE + PPP liabilities (% GDP)
      primary_balance      — primary fiscal balance (% GDP)
    """
    rng = np.random.default_rng(seed)

    # Latent fiscal stress dimension drives indicators
    F_stress = rng.normal(0, 1, N)         # higher = more stressed
    F_struct  = rng.normal(0, 1, N)        # structural revenue capacity

    e = lambda s: rng.normal(0, s, N)

    debt_to_gdp           = 55  + 18 * F_stress - 5  * F_struct + e(8)
    fiscal_deficit        = -3  -  2 * F_stress + 0.8 * F_struct + e(1)
    interest_to_revenue   = 18  +  6 * F_stress - 2  * F_struct + e(3)
    external_debt_share   = 50  + 10 * F_stress - 3  * F_struct + e(6)
    revenue_to_gdp        = 18  -  3 * F_stress + 4  * F_struct + e(2)
    grants_dependency     = 12  +  4 * F_stress - 2  * F_struct + e(3)
    debt_service_ratio    = 15  +  5 * F_stress - 1  * F_struct + e(2.5)
    contingent_liabilities= 8   +  3 * F_stress + 0.5 * e(2)
    primary_balance       = -1  -  1.5 * F_stress + 0.5 * F_struct + e(0.8)

    data = {
        'debt_to_gdp':            np.clip(debt_to_gdp,           15, 130),
        'fiscal_deficit':         np.clip(fiscal_deficit,        -15,   3),
        'interest_to_revenue':    np.clip(interest_to_revenue,     3,  45),
        'external_debt_share':    np.clip(external_debt_share,    10,  90),
        'revenue_to_gdp':         np.clip(revenue_to_gdp,          8,  35),
        'grants_dependency':      np.clip(grants_dependency,       0,  40),
        'debt_service_ratio':     np.clip(debt_service_ratio,      2,  35),
        'contingent_liabilities': np.clip(contingent_liabilities,  0,  20),
        'primary_balance':        np.clip(primary_balance,        -9,   3),
    }
    return pd.DataFrame(data, index=countries)


df = generate_fiscal_data(COUNTRIES)
INDICATOR_COLS = df.columns.tolist()

print(f"Dataset : {df.shape[0]} countries  x  {len(INDICATOR_COLS)} fiscal indicators")
print(df.describe().round(2).to_string())

# Visualise indicator distributions
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
fig.suptitle('Fiscal Vulnerability Indicators — Distribution across African Economies',
             fontsize=12, fontweight='bold')
for ax, col in zip(axes.flatten(), INDICATOR_COLS):
    ax.hist(df[col], bins=20, color='steelblue', alpha=0.75, edgecolor='white')
    ax.set_title(col.replace('_', ' ').title(), fontsize=9)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lecture18_exercise_distributions.png', bbox_inches='tight')
plt.show()
print("Saved : lecture18_exercise_distributions.png")

## ── SECTION 3: STANDARDISE FEATURES (provided)

In [ ]:
scaler = StandardScaler()
X_sc   = scaler.fit_transform(df[INDICATOR_COLS].values)
print("\nFeatures standardised.")

## ── SECTION 4: FIT PCA & SCREE PLOT  ◄─ YOUR TASK (TODO 1)

1. Fit a full PCA on X_sc:

pca_full = PCA(random_state=SEED)

pca_full.fit(X_sc)

2. Extract:

evr     = pca_full.explained_variance_ratio_

cum_evr = np.cumsum(evr)

3. Print a variance table (PC, eigenvalue, variance %, cumulative %).

4. Plot a scree plot (individual and cumulative) and mark the 90% threshold.

Also mark how many components are needed to reach 90%.

In [ ]:
# WRITE YOUR CODE BELOW
# pca_full = PCA(random_state=SEED)
# pca_full.fit(X_sc)
# evr      = pca_full.explained_variance_ratio_
# cum_evr  = np.cumsum(evr)
# ...

## ── SECTION 5: INTERPRET LOADINGS  ◄─ YOUR TASK (TODO 2)

1. Fit PCA keeping 3 components:

pca3   = PCA(n_components=3, random_state=SEED)

scores = pca3.fit_transform(X_sc)    # shape (45, 3)

2. Build a loadings DataFrame:

loadings = pd.DataFrame(

pca3.components_.T,

index=INDICATOR_COLS,

columns=['PC1', 'PC2', 'PC3']

)

print(loadings.round(3))

3. Plot a heatmap of the loadings.

4. Answer in comments:

- What does PC1 capture? (hint: look at the largest absolute loadings)

- Which indicator has the highest positive loading on PC1?

- Is a HIGH PC1 score associated with HIGH or LOW fiscal vulnerability?

In [ ]:
# WRITE YOUR CODE BELOW
# pca3   = PCA(n_components=3, random_state=SEED)
# scores = pca3.fit_transform(X_sc)
# loadings = ...
# ...

# Your interpretation (fill in):
# PC1 captures : ...
# High PC1 score means : ...

## ── SECTION 6: CONSTRUCT FISCAL VULNERABILITY INDEX  ◄─ YOUR TASK (TODO 3)

1. Extract PC1 scores:  fvi_raw = scores[:, 0]

2. Flip the sign if needed so that a HIGHER score = MORE vulnerable.

(Hint: check the sign of the loading for 'debt_to_gdp' on PC1.

If it is negative, multiply fvi_raw by -1.)

3. Rescale to 0–100:

fvi_min, fvi_max = fvi_raw.min(), fvi_raw.max()

fvi = 100 * (fvi_raw - fvi_min) / (fvi_max - fvi_min)

So that 100 = most vulnerable, 0 = least vulnerable.

4. Add to df:  df['fiscal_vulnerability_index'] = fvi

In [ ]:
# WRITE YOUR CODE BELOW
# fvi_raw = scores[:, 0]
# ...
# df['fiscal_vulnerability_index'] = fvi

## ── SECTION 7: RANK & VISUALISE  ◄─ YOUR TASK (TODO 4)

1. Sort df by 'fiscal_vulnerability_index' descending (most vulnerable first).

2. Plot a horizontal bar chart showing all 45 countries ranked by FVI.

- Use a diverging colormap (e.g., RdYlGn_r) to highlight vulnerability.

- Label each bar with its FVI value.

- Title: 'Fiscal Vulnerability Index — African Economies (PCA-Derived)'

3. Print the top-10 most vulnerable and top-10 least vulnerable economies.

Starter snippet:

ranking = df[['fiscal_vulnerability_index']].sort_values(

'fiscal_vulnerability_index', ascending=False)

In [ ]:
# WRITE YOUR CODE BELOW
# ranking = df[['fiscal_vulnerability_index']].sort_values(
#     'fiscal_vulnerability_index', ascending=False)
# ...

## ── SECTION 8: DISCUSSION QUESTIONS

In [ ]:
print("""
── Discussion Questions ───────────────────────────────────────────────────────
 1. How many principal components did you need to capture 90 % of the variance?
    What does this tell you about the dimensionality of fiscal vulnerability?

 2. PC1 loading signs determine whether a country with high debt scores high
    or low on your index. Did you need to flip the sign? How did you decide?

 3. Compare the top-5 most vulnerable economies to what you know about
    actual African fiscal conditions. Does the ranking make intuitive sense?

 4. What are the limitations of using only PC1 to construct the FVI?
    How could you incorporate PC2 and PC3?

 5. A finance minister asks: "What single indicator most explains your
    vulnerability ranking?" What would you say, and why?
──────────────────────────────────────────────────────────────────────────────
""")

print("=== EXERCISE COMPLETE ===")